# 🎙️ Voice Product Discovery — one-click Colab launch

Runs the whole app (FastAPI + LangGraph + MCP + Whisper + TTS + React UI) on this Colab VM and exposes it at a public **HTTPS** URL, so the microphone works in your browser.

**Before you run:**
1. *(Recommended)* Click the **🔑 key icon** in the left sidebar → **Add new secret** → name it `OPENAI_API_KEY`, paste your key, and enable **Notebook access**. Your key stays in *your* Google account — it is never written to the repo or shown to other users.
2. No key? It still works: the notebook falls back to a keyless **mock LLM** (deterministic demo heuristics, clearly labeled in the agent step log).
3. In the first code cell, `REPO_URL` must point at your GitHub copy of this repo (edit it once before committing this notebook). Private repo? Add a `GITHUB_TOKEN` secret too.

Then **Runtime → Run all** and open the URL printed by the last cell. First full setup takes ~4–6 minutes.

In [ ]:
REPO_URL = "https://github.com/aimanaltoubi/voice-product-discovery.git"  # @param {type:"string"}

import pathlib, re, subprocess
assert "YOUR-USERNAME" not in REPO_URL, "Edit REPO_URL above to point at your GitHub repo."
name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
clone_url = REPO_URL
try:  # optional: GITHUB_TOKEN secret for private repos
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        clone_url = re.sub(r"^https://", f"https://{tok}@", REPO_URL)
except Exception:
    pass
if not pathlib.Path(name).exists():
    subprocess.run(["git", "clone", "--depth", "1", clone_url, name], check=True)
%cd {name}
REPO = pathlib.Path.cwd()
print("Repo ready at", REPO)

In [ ]:
%%bash
# Node 18+ for the Vite build, plus all backend Python deps.
set -e
if ! node -e 'process.exit(parseInt(process.versions.node)>=18?0:1)' 2>/dev/null; then
  echo "Installing Node 20…"
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
  apt-get install -y nodejs > /dev/null 2>&1
fi
echo "node $(node --version)"
echo "Installing backend requirements (a few minutes)…"
pip install -q -r backend/requirements.txt
echo "Backend deps installed."

In [ ]:
# Provider config — reads OPENAI_API_KEY from Colab Secrets, mock fallback.
import os
key = None
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
except Exception:
    key = None
if key:
    os.environ["OPENAI_API_KEY"] = key
    os.environ["LLM_PROVIDER"] = "openai"
    os.environ.setdefault("LLM_MODEL", "gpt-4o-mini")
    print("✅ OPENAI_API_KEY found in Colab Secrets → real LLM mode (" + os.environ["LLM_MODEL"] + ").")
else:
    os.environ["LLM_PROVIDER"] = "mock"
    print("⚠️ No OPENAI_API_KEY secret → keyless MOCK mode (deterministic demo heuristics).")
    print("   Add the secret via the 🔑 sidebar, enable Notebook access, and re-run this cell + the server cell.")
os.environ.setdefault("EMBEDDINGS_PROVIDER", "local")  # keyless ONNX MiniLM
os.environ.setdefault("ASR_PROVIDER", "local")         # faster-whisper on CPU
os.environ.setdefault("TTS_PROVIDER", "edge")          # keyless Edge voices
print("embeddings=local · asr=local(faster-whisper) · tts=edge")

In [ ]:
# Build the private-catalog index (bundled 24-product sample).
# To use the real Kaggle Amazon-2020 slice instead, see data/README.md.
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "rag.ingest", "--sample"],
               cwd=str(REPO / "backend"), env=os.environ, check=True)
print("Index built.")

In [ ]:
%%bash
set -e
cd frontend
echo "Installing frontend deps…"
npm ci --silent 2>/dev/null || npm install --silent
npx vite build
echo "UI built → frontend/dist"

In [ ]:
# Start the single-port server (UI + /api + /media on :8000).
import os, subprocess, sys, time, urllib.request
try:
    server.kill()  # re-running this cell restarts the server
except NameError:
    pass
server = subprocess.Popen(
    [sys.executable, "scripts/serve_colab.py"],
    cwd=str(REPO), env=os.environ,
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
ok = False
for _ in range(60):
    try:
        body = urllib.request.urlopen("http://localhost:8000/api/health", timeout=2).read().decode()
        print("Backend healthy:", body[:130], "…")
        ok = True
        break
    except Exception:
        time.sleep(2)
if not ok:
    print(open("/content/server.log").read()[-4000:])
    raise RuntimeError("Backend did not start — see log above.")

In [ ]:
# Public HTTPS tunnel (Cloudflare quick tunnel — no account needed).
# HTTPS is required for the browser microphone to work.
import re, subprocess, time
subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
               check=True)
subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)
try:
    tunnel.kill()
except NameError:
    pass
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url, lines, t0 = None, [], time.time()
while time.time() - t0 < 90 and url is None:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    lines.append(line)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
if url:
    print("\n" + "=" * 72)
    print(f"  🎉 YOUR APP IS LIVE:   {url}")
    print("=" * 72)
    print("Open it, allow the microphone, and speak. Keep this notebook running.")
    print("First transcription downloads the Whisper model once (~150 MB), so it's slow one time.")
else:
    print("".join(lines[-30:]))
    raise RuntimeError("Tunnel URL not found — see cloudflared output above; re-run this cell.")

### Notes & troubleshooting

- **Try saying:** “Find me an eco-friendly stainless-steel cleaner under fifteen dollars” · “What's the current price of a glass cleaner right now?” · “Can I mix bleach and ammonia?” (safety demo)
- The URL changes every session and dies when the notebook disconnects (Colab idle limits apply) — this is a demo runtime, not hosting.
- Added the key after starting? Re-run the **provider config** cell, then the **server** cell (the tunnel cell can stay).
- Backend logs: `/content/server.log` · agent run logs: `backend/logs/runs/` · MCP tool logs: `backend/logs/mcp_server.jsonl`.
- Mock mode is a deterministic heuristic, not a language model — use your key for real answer quality.